# Flipkart Gridlock 2.0: V14 Advanced Graph-Residual Cascade
## Asymmetric Spatial Graph Modeling on Microscopic Anomaly Deltas

**Architectural Paradigm:**
This pipeline represents the convergence of Version 13 and Version 14. Stage 1 isolates the macroscopic historical baseline via Out-of-Fold Target Encoding. Stage 2 constructs an asymmetric spatial graph across the resulting residuals—calculating neighbor spillover and velocity vectors entirely on the error terms. The base engines are optimized via asymmetric loss gradients to focus 100% of their capacity on unmapped spatiotemporal anomalies.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import pygeohash as pgh
from scipy.optimize import minimize
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
np.random.seed(42)

base_paths = [".", "data/raw", "../../data/raw"]
base_path = next((path for path in base_paths if os.path.exists(os.path.join(path, "train.csv"))), None)

raw_train = pd.read_csv(os.path.join(base_path, "train.csv"))
raw_test = pd.read_csv(os.path.join(base_path, "test.csv"))

y_train_raw = raw_train['demand'].values
submission_index = raw_test['Index'].values

# Stage 1: Build the Microscopic Baseline Map First
def apply_oof_encoding(tr_df, te_df, tgt, col, folds=5):
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)
    tr_enc = np.zeros(len(tr_df))
    tmp_tr = tr_df[[col]].copy()
    tmp_tr['tgt'] = tgt
    g_mean = tgt.mean()
    
    for tr_idx, val_idx in kf.split(tmp_tr):
        f_map = tmp_tr.iloc[tr_idx].groupby(col)['tgt'].mean()
        tr_enc[val_idx] = tmp_tr.iloc[val_idx][col].map(f_map).fillna(g_mean).values
        
    te_map = tmp_tr.groupby(col)['tgt'].mean()
    te_enc = te_df[col].map(te_map).fillna(g_mean).values
    return tr_enc, te_enc

# Construct the interaction key for baseline isolation
t_split_tr = raw_train['timestamp'].str.split(':', expand=True).astype(int)
ts_min_tr = t_split_tr[0] * 60 + t_split_tr[1]
raw_train['time_slot_15m'] = ts_min_tr // 15
raw_train['geo_time_interaction'] = raw_train['geohash'].astype(str) + "_" + raw_train['time_slot_15m'].astype(str)

t_split_te = raw_test['timestamp'].str.split(':', expand=True).astype(int)
ts_min_te = t_split_te[0] * 60 + t_split_te[1]
raw_test['time_slot_15m'] = ts_min_te // 15
raw_test['geo_time_interaction'] = raw_test['geohash'].astype(str) + "_" + raw_test['time_slot_15m'].astype(str)

baseline_train, baseline_test = apply_oof_encoding(raw_train, raw_test, y_train_raw, 'geo_time_interaction')

# CRITICAL STEP: Extract Pure Residual targets for Stage 2
y_train_residual = y_train_raw - baseline_train
raw_train['residual_demand'] = y_train_residual

print("Stage 1 Baseline complete. Deep residuals isolated for Graph Processing.")

Stage 1 Baseline complete. Deep residuals isolated for Graph Processing.


In [2]:
def engineer_graph_residual_features(source_df, target_df):
    df_f = target_df.copy()
    
    # Autoregressive Lag Matrices tracking RESIDUALS instead of raw numbers
    lag_df = source_df[['geohash', 'day', 'timestamp', 'residual_demand']].copy()
    
    lag_24 = lag_df.copy()
    lag_24['day'] += 1
    lag_24.rename(columns={'residual_demand': 'res_lag_24h'}, inplace=True)
    
    lag_48 = lag_df.copy()
    lag_48['day'] += 2
    lag_48.rename(columns={'residual_demand': 'res_lag_48h'}, inplace=True)
    
    df_f = df_f.merge(lag_24, on=['geohash', 'day', 'timestamp'], how='left')
    df_f = df_f.merge(lag_48, on=['geohash', 'day', 'timestamp'], how='left')
    df_f[['res_lag_24h', 'res_lag_48h']] = df_f[['res_lag_24h', 'res_lag_48h']].fillna(0.0)
    
    # Anomaly Momentum Vector
    df_f['res_momentum'] = df_f['res_lag_24h'] - df_f['res_lag_48h']
    
    # Graph Adjacency Mapping across Error Terms
    res_24_lookup = lag_24.set_index(['geohash', 'day', 'timestamp'])['res_lag_24h'].to_dict()
    
    def compute_residual_spillover(row):
        try:
            neighbors = pgh.neighbors(row['geohash'])
            spillover_sum = 0.0
            valid_nodes = 0
            for n in neighbors:
                key = (n, row['day'], row['timestamp'])
                if key in res_24_lookup:
                    spillover_sum += res_24_lookup[key]
                    valid_nodes += 1
            return spillover_sum / valid_nodes if valid_nodes > 0 else 0.0
        except Exception:
            return 0.0
            
    print("Tracing Anomaly Waves across the Adjacency Graph...")
    df_f['neighbor_residual_spillover'] = df_f.apply(compute_residual_spillover, axis=1)
    
    # Kinematics
    df_f['hour'] = df_f['timestamp'].str.split(':', expand=True)[0].astype(int)
    df_f['hour_sin'] = np.sin(2 * np.pi * df_f['hour'] / 24.0)
    df_f['hour_cos'] = np.cos(2 * np.pi * df_f['hour'] / 24.0)
    df_f['is_rush_hour'] = df_f['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
    
    df_f['Temperature'] = df_f['Temperature'].fillna(df_f['Temperature'].median())
    for col in ['Weather', 'RoadType', 'LargeVehicles', 'Landmarks']:
        if col in df_f.columns:
            df_f[col] = df_f[col].fillna('Unknown')
            
    return df_f

X_train_fe = engineer_graph_residual_features(raw_train, raw_train.drop(columns=['demand', 'residual_demand'], errors='ignore'))
X_test_fe = engineer_graph_residual_features(raw_train, raw_test.drop(columns=['Index'], errors='ignore'))

X_train_fe['Baseline_TE'] = baseline_train
X_test_fe['Baseline_TE'] = baseline_test

cat_features = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
for c in cat_features:
    le = LabelEncoder()
    le.fit(X_train_fe[c].astype(str).tolist() + X_test_fe[c].astype(str).tolist())
    X_train_fe[c] = le.transform(X_train_fe[c].astype(str))
    X_test_fe[c] = le.transform(X_test_fe[c].astype(str))

drop_cols = ['timestamp', 'geo_time_interaction', 'Index']
features = [c for c in X_train_fe.columns if c not in drop_cols]

X = X_train_fe[features].values
X_test = X_test_fe[features].values
X_test = np.nan_to_num(X_test, nan=0.0)

# Custom Asymmetric Engine for Residual Optimization
def asymmetric_residual_mse(preds, train_data):
    y_true = train_data.get_label()
    residual = (y_true - preds).astype("float")
    grad = np.where(residual > 0, -2.0 * 1.4 * residual, -2.0 * residual)
    hess = np.where(residual > 0, 2.0 * 1.4, 2.0)
    return grad, hess

kf = KFold(n_splits=5, shuffle=True, random_state=42)

lgb_params = {'objective': asymmetric_residual_mse, 'metric': 'rmse', 'learning_rate': 0.02, 'max_depth': 8, 'num_leaves': 64, 'min_child_samples': 20, 'verbose': -1, 'random_state': 42, 'n_jobs': -1}
xgb_params = {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'learning_rate': 0.02, 'max_depth': 6, 'random_state': 42, 'n_jobs': -1}
cat_params = {'iterations': 2500, 'learning_rate': 0.02, 'depth': 7, 'eval_metric': 'RMSE', 'verbose': 0, 'random_seed': 42}

oof_res_lgb, test_res_lgb = np.zeros(len(X)), np.zeros(len(X_test))
oof_res_xgb, test_res_xgb = np.zeros(len(X)), np.zeros(len(X_test))
oof_res_cat, test_res_cat = np.zeros(len(X)), np.zeros(len(X_test))

print("Executing Stage 2 Residual Graph Triad Training...")
for fold, (t_idx, v_idx) in enumerate(kf.split(X)):
    X_tr, y_tr = X[t_idx], y_train_residual[t_idx]
    X_va, y_va = X[v_idx], y_train_residual[v_idx]
    
    m_lgb = lgb.train(lgb_params, lgb.Dataset(X_tr, y_tr), num_boost_round=3000, valid_sets=[lgb.Dataset(X_va, y_va)], callbacks=[lgb.early_stopping(150, verbose=False)])
    oof_res_lgb[v_idx] = m_lgb.predict(X_va)
    test_res_lgb += m_lgb.predict(X_test) / 5
    
    m_xgb = xgb.train(xgb_params, xgb.DMatrix(X_tr, y_tr), 3000, evals=[(xgb.DMatrix(X_va, y_va), 'val')], early_stopping_rounds=150, verbose_eval=False)
    oof_res_xgb[v_idx] = m_xgb.predict(xgb.DMatrix(X_va))
    test_res_xgb += m_xgb.predict(xgb.DMatrix(X_test)) / 5
    
    m_cat = CatBoostRegressor(**cat_params).fit(X_tr, y_tr, eval_set=(X_va, y_va))
    oof_res_cat[v_idx] = m_cat.predict(X_va)
    test_res_cat += m_cat.predict(X_test) / 5
    
    print(f"Fold {fold+1} residual validation finalized.")

# Global Optimization Against TRUE Target Raw Demand
def objective_reconstruction(weights):
    w = np.array(weights)
    if w.sum() == 0: return 999.0
    w_norm = w / w.sum()
    blended_residual = (w_norm[0] * oof_res_lgb) + (w_norm[1] * oof_res_xgb) + (w_norm[2] * oof_res_cat)
    reconstructed_oof = np.clip(baseline_train + blended_residual, 0.0, 1.0)
    return -max(0, 100 * r2_score(y_train_raw, reconstructed_oof))

optimal_weights = minimize(objective_reconstruction, [0.38, 0.29, 0.33], method='Nelder-Mead').x
optimal_weights /= sum(optimal_weights)

final_blended_residual = (optimal_weights[0] * test_res_lgb) + (optimal_weights[1] * test_res_xgb) + (optimal_weights[2] * test_res_cat)
final_test_predictions = np.clip(baseline_test + final_blended_residual, 0.0, 1.0)

final_oof_residual = (optimal_weights[0] * oof_res_lgb) + (optimal_weights[1] * oof_res_xgb) + (optimal_weights[2] * oof_res_cat)
reconstructed_oof = np.clip(baseline_train + final_oof_residual, 0.0, 1.0)
final_r2 = max(0, 100 * r2_score(y_train_raw, reconstructed_oof))

pd.DataFrame({'Index': submission_index, 'demand': final_test_predictions}).to_csv("submission_v14.csv", index=False)

print("\n==================================================")
print("PIPELINE EXECUTION COMPLETE (V14.5 HYBRID)")
print("==================================================")
print(f"Optimal Weights: LGBM: {optimal_weights[0]:.3f} | XGB: {optimal_weights[1]:.3f} | CAT: {optimal_weights[2]:.3f}")
print(f"Terminal Reconstructed R2 Score: {final_r2:.4f}")
print("Output Matrix Generated: submission_v14.csv")

Tracing Anomaly Waves across the Adjacency Graph...
Tracing Anomaly Waves across the Adjacency Graph...
Executing Stage 2 Residual Graph Triad Training...
Fold 1 residual validation finalized.
Fold 2 residual validation finalized.
Fold 3 residual validation finalized.
Fold 4 residual validation finalized.
Fold 5 residual validation finalized.

PIPELINE EXECUTION COMPLETE (V14.5 HYBRID)
Optimal Weights: LGBM: 0.427 | XGB: 0.449 | CAT: 0.124
Terminal Reconstructed R2 Score: 88.1755
Output Matrix Generated: submission_v14.csv
